In [26]:
import numpy as np
import joblib
import json
import tensorflow as tf
from sentence_transformers import SentenceTransformer

from keras.models import load_model

model = SentenceTransformer('all-MiniLM-L6-v2')

print("Embedding working ✅")

c:\Users\shrey\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Embedding working ✅


In [27]:
autoencoder = load_model("../models/autoencoder.h5", compile=False)
iso_forest = joblib.load("../models/isolation_forest.pkl")

print("Models loaded")

Models loaded


In [28]:
x_test_scaled = np.load("X_test_scaled.npy")
y_test = np.load("y_test.npy")

print("Data loaded:", x_test_scaled.shape)

Data loaded: (25192, 116)


In [29]:
# Autoencoder
reconstructions = autoencoder.predict(x_test_scaled)
mse = np.mean((x_test_scaled - reconstructions)**2, axis=1)

# Isolation Forest
if_scores = -iso_forest.decision_function(x_test_scaled)

# Normalize
from sklearn.preprocessing import MinMaxScaler

ae_norm = MinMaxScaler().fit_transform(mse.reshape(-1,1)).flatten()
if_norm = MinMaxScaler().fit_transform(if_scores.reshape(-1,1)).flatten()

# Hybrid score
alpha = 0.6
hybrid_score = alpha * ae_norm + (1 - alpha) * if_norm

# Threshold (reuse or recompute)
threshold = np.percentile(hybrid_score, 80)

y_pred = (hybrid_score > threshold).astype(int)

print("Predictions ready")

788/788 ━━━━━━━━━━━━━━━━━━━━ 1s 915us/step
Predictions ready


In [37]:
sample_index = 13  # you can change this

print("Prediction:", y_pred[sample_index])

Prediction: 0


In [39]:
sample = x_test_scaled[sample_index]
recon = reconstructions[sample_index]

feature_error = np.abs(sample - recon)

top_indices = np.argsort(feature_error)[-3:][::-1]

# Load feature names (IMPORTANT)
feature_names = np.load("feature_names.npy", allow_pickle=True)

top_features = [feature_names[i] for i in top_indices]

print("Top features:", top_features)

Top features: ['service_ftp_data', 'dst_host_same_src_port_rate', 'dst_host_srv_diff_host_rate']


In [40]:
with open("knowledge_base.json", "r") as f:
    knowledge_base = json.load(f)

documents = [
    entry["description"] + " " + " ".join(entry["indicators"])
    for entry in knowledge_base
]

In [41]:
import faiss 
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

embeddings = embedding_model.encode(documents)

dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)
index.add(np.array(embeddings))

print("Knowledge base indexed")

Knowledge base indexed


In [42]:
query = "high " + " high ".join(top_features)

query_embedding = embedding_model.encode([query])

D, I = index.search(np.array(query_embedding), k=2)

retrieved_docs = [documents[i] for i in I[0]]

print("Retrieved knowledge:")
print(retrieved_docs)


Retrieved knowledge:
['Port scanning involves probing multiple ports on a host to identify open services. high dst_host_count high diff_srv_rate low duration', 'Normal network traffic follows expected patterns with low error rates and consistent behavior. balanced src_bytes low error rates stable connection patterns']


In [43]:
def generate_explanation(features, docs):

    explanation = f"""
🚨 Intrusion Detection Alert

Top anomalous features:
{', '.join(features)}

Analysis:
The network traffic shows unusual behavior based on the detected features.
{docs[0]}

Conclusion:
This activity is likely malicious and should be investigated further.
"""

    return explanation

In [44]:
explanation = generate_explanation(top_features, retrieved_docs)

print(explanation)


🚨 Intrusion Detection Alert

Top anomalous features:
service_ftp_data, dst_host_same_src_port_rate, dst_host_srv_diff_host_rate

Analysis:
The network traffic shows unusual behavior based on the detected features.
Port scanning involves probing multiple ports on a host to identify open services. high dst_host_count high diff_srv_rate low duration

Conclusion:
This activity is likely malicious and should be investigated further.

